## NF-CORE/MAG setup for "full" pipeline and "assembled" partial execution 

There are two ways to run the nf-core/mag workflow: either starting from the raw (adaptor-removed and cleaned) sequences (the same ones we get back from Genoscope), or starting from assembled sequences (the MEGAHIT final.contigs.fa sequences from MGF). If using the "assembled" partial workflow, then the initial cleaning, trimming, and assembly steps of the full workflow are skipped.

### Setup

```
.
├── custom.config
├── data
│   ├── HCFCYDSX5.UDI129
│   ├── HCFCYDSX5.UDI135
│   └── HJWK3DSX7.UDI362
├── libs
│   ├── gtdbtk_r220_data.tar.gz
│   └── gtdbtk_r220_data.tar.gz.1
├── results
│   ├── assembly
│   └── full
├── run-mag-assembled.sh
├── run-mag-full.sh
├── UDI362mf.err
├── UDI362mf.log
└── work

```

- `custom.config` - workflow parameters overriding defaults
- `/data` - data directory
- `/libs` - other cached libraries (may or may not be used)
- `/results`
- `/results/assembled` - where the results of the "assembled" wf go
- `/results/full` - where the results of the "full" workflow go
- `/work`- wf directory where temporary directories are place (do not touch)


**NB as of 19/05/2025 because the db is downloaded each time the wf runs and keeps failing. Currently download times from the server are in the 400 days (yes days), so we can use directly by using the `--gtdb_db libs/gtdbtk_r220_data.tar.gz` flag**

##### Tags

```
UDI129ma - assembled wf
```

```
UDI129mf - full wf
```

### Data files

##### Full worklow
Only requires the raw sequences from Genoscope:
```
redi:/usr/local/scratch/emo-bon-sequencing-data/www.genoscope.cns.fr/sadc/projet_DBB
```

##### Assembled workflow
Requires the same two raw sequence data files from Genoscope, plus the assembled contigs from MGF. The assembled contigs from MGF are all named the same in very MGF results: final.contig.fa

**NB the contigs need to be gzipped**

```
$ gzip -c final.contigs.fa > final.contigs.fa.gz
```


### Custom parameters
```
$ cat custom-bigmem.config 
process {
    resourceLimits = [
        cpus: 54,
        memory: 110.GB,
        time: 96.h
    ]
    withName: METASPADES {
        cpus = 54,
        memory = 800.GB
    }
    withName: MEGAHIT {
        time = 48.h
    }
    withName: BOWTIE2_ASSEMBLY_ALIGN {
        time = 48.h
    }
}
```
METASPADES won't assemble an EMO BON sample with <120GB RAM!

### Full workflow using METASPADES on ceta-gen (big-mem)

**This nf-core/mag wf does contig assembly using METASPADES and binning using MAXBIN2 only running on ceta-gen!**

Requires an input spreadsheet only:

e.g
```
$ pwd
/home/cymon/src/nf-core-mag/data/HJWK3DSX7.UDI362
$ ls
DBH_AACVOSDA_2_1_HJWK3DSX7.UDI362_clean.fastq.gz  DBH_AACVOSDA_2_2_HJWK3DSX7.UDI362_clean.fastq.gz  input.csv
$ cat input.csv 
sample,group,short_reads_1,short_reads_2,long_reads
UDI362,0,data/HJWK3DSX7.UDI362/DBH_AACVOSDA_2_1_HJWK3DSX7.UDI362_clean.fastq.gz,data/HJWK3DSX7.UDI362/DBH_AACVOSDA_2_2_HJWK3DSX7.UDI362_clean.fastq.gz,
```

Submission script in top-level dir:
```
$ cat UDI362mf-sbatch.sh
#!/bin/bash

#SBATCH --nodes=1
#SBATCH --nodelist=ceta-gen
#SBATCH --partition=bigmem
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=54
#SBATCH --job-name=UDI362mf
#SBATCH --output=UDI362mf.log
#SBATCH --error=UDI362mf.err

export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-17.0.15.0.6-2.el8.x86_64
export PATH=$JAVA_HOME/bin:$PATH
export NXF_APPTAINER_CACHEDIR=/share/apps/share/nextflow/apptainer_cache

date
nextflow -log results/full/HJWK3DSX7.UDI362/nexflow.log \
    run nf-core/mag -r 3.4.0 \
    -c custom-bigmem.config \
    -profile apptainer \
    --input data/HJWK3DSX7.UDI362/input.csv \
    --outdir results/full/HJWK3DSX7.UDI362 \
    --gtdb_db libs/gtdbtk_r202_data.tar.gz \
    --skip_megahit \
    --skip_concoct \
    --skip_metabat2

#    -resume
#    --skip_gtdbtk \
date

```



#### FULL SLURM Submission

##### Bash script code
```
$ cat write-slurm-full-script.sh 
#!/bin/bash

DATA_FORWARD=$1
NUM=`expr substr $DATA_FORWARD 14 1`
if [[ "$NUM" = 1 ]]; then
    DATA_REVERSE=${DATA_FORWARD/_1_1_/_1_2_}
elif [[ "$NUM" = 2 ]]; then
    DATA_REVERSE=${DATA_FORWARD/_2_1_/_2_2_}
elif [[ "$NUM" = 3 ]]; then
    DATA_REVERSE=${DATA_FORWARD/_3_1_/_3_2_}
fi
RUN_CODE=${DATA_FORWARD:17:16}
UDI=${RUN_CODE:10:6}
SUBFILE=${UDI}mf-sbatch.sh
INPUT_FILEPATH="data/${RUN_CODE}/input.csv"

echo "DATA_FORWARD = $DATA_FORWARD"
echo "DATA_REVERSE = $DATA_REVERSE"
echo "RUN_CODE = $RUN_CODE"
echo "UDI = $UDI"
echo "SUBFILE = $SUBFILE"
echo "INPUT_FILEPATH = $INPUT_FILEPATH"

cat > $SUBFILE <<EOF
#!/bin/bash

#SBATCH --nodes=1
#SBATCH --nodelist=ceta-gen
#SBATCH --partition=bigmem
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=54
#SBATCH --job-name=${UDI}mf
#SBATCH --output=${UDI}mf.log
#SBATCH --error=${UDI}mf.err

export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-17.0.15.0.6-2.el8.x86_64
export PATH=\$JAVA_HOME/bin:\$PATH
export NXF_APPTAINER_CACHEDIR=/share/apps/share/nextflow/apptainer_cache

date
nextflow -log results/full/${RUN_CODE}/nexflow.log \\
    run nf-core/mag -r 3.4.0 \\
    -c custom-bigmem.config \\
    -profile apptainer \\
    --input data/${RUN_CODE}/input.csv \\
    --outdir results/full/${RUN_CODE} \\
    --gtdb_db libs/gtdbtk_r202_data.tar.gz \\
    --skip_megahit \\
    --skip_concoct \\
    --skip_metabat2

#    -resume
#    --skip_gtdbtk \\
date
EOF

cat > $INPUT_FILEPATH <<EOF
sample,group,short_reads_1,short_reads_2,long_reads
${UDI},0,data/${RUN_CODE}/${DATA_FORWARD},data/${RUN_CODE}/${DATA_REVERSE},
EOF

echo
echo "Input file:"
cat $INPUT_FILEPATH
echo
echo "Submission script:"
cat $SUBFILE
echo
echo
echo "This nf-core/mag wf does contig assembly using METASPADES and binning using MAXBIN2 only running on ceta-gen!"

```
##### Usage

```
$ ./write-slurm-full-script.sh DBH_AABAOSDA_1_1_HCFCYDSX5.UDI129_clean.fastq.gz
```

### Assembled workflow

**This nf-core/mag wf uses pre-computed MEGAHIT contigs and does binning using MAXBIN2 only**

Requires two input files and the same raw sequences:

```
$ ll
total 14555948
-rw-rw-r-- 1 cymon cymon         84 May 18 12:58 assembled_input.csv
-rw-r--r-- 1 cymon cymon 7186776593 May 18 13:51 DBH_AABAOSDA_1_1_HCFCYDSX5.UDI129_clean.fastq.gz
-rw-r--r-- 1 cymon cymon 7502611324 May 18 13:53 DBH_AABAOSDA_1_2_HCFCYDSX5.UDI129_clean.fastq.gz
-rw-rw-r-- 1 cymon cymon  215879051 May 18 12:27 final.contigs.fa.gz
-rw-rw-r-- 1 cymon cymon        205 May 18 13:52 input.csv
$ cat input.csv 
sample,group,short_reads_1,short_reads_2,long_reads
UDI129,0,data/HCFCYDSX5.UDI129/DBH_AABAOSDA_1_1_HCFCYDSX5.UDI129_clean.fastq.gz,data/HCFCYDSX5.UDI129/DBH_AABAOSDA_1_2_HCFCYDSX5.UDI129_clean.fastq.gz,

$ cat assembled_input.csv 
id,group,assembler,fasta
UDI129,0,MEGAHIT,data/HCFCYDSX5.UDI129/final.contigs.fa.gz

```

Submission script in top dir:

```
$ cat UDI135ma-sbatch.sh
#!/bin/bash

##SBATCH --exclude=ceta3,ceta4,ceta5,ceta6,ceta-gen,ceta.ualg.pt
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=54
#SBATCH --job-name=UDI135ma
#SBATCH --output=UDI135ma.log
#SBATCH --error=UDI135ma.err

export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-17.0.15.0.6-2.el8.x86_64
export PATH=$JAVA_HOME/bin:$PATH
export NXF_APPTAINER_CACHEDIR=/share/apps/share/nextflow/apptainer_cache

date
nextflow -log results/assembled/HCFCYDSX5.UDI135/nexflow.log \
    run nf-core/mag -r 3.4.0 \
    -c custom.config \
    -profile apptainer \
    --assembly_input data/HCFCYDSX5.UDI135/assembled_input.csv \
    --input data/HCFCYDSX5.UDI135/input.csv \
    --outdir results/assembled/HCFCYDSX5.UDI135 \
    --gtdb_db libs/gtdbtk_r202_data.tar.gz \
    --skip_concoct \
    --skip_metabat2

#    --skip_gtdbtk
#    -resume
date
```



#### Assembled SLURM submission: automated scripting of assembled input sheets and submission script

##### Bash script code:

```
$ cat write-slurm-assembled-script.sh 
#!/bin/bash

DATA_FORWARD=$1
NUM=`expr substr $DATA_FORWARD 14 1`
if [[ "$NUM" = 1 ]]; then
    DATA_REVERSE=${DATA_FORWARD/_1_1_/_1_2_}
elif [[ "$NUM" = 2 ]]; then
    DATA_REVERSE=${DATA_FORWARD/_2_1_/_2_2_}
elif [[ "$NUM" = 3 ]]; then
    DATA_REVERSE=${DATA_FORWARD/_3_1_/_3_2_}
fi
RUN_CODE=${DATA_FORWARD:17:16}
UDI=${RUN_CODE:10:6}
SUBFILE=${UDI}ma-sbatch.sh
ASS_INPUT_FILEPATH="data/${RUN_CODE}/assembled_input.csv"
INPUT_FILEPATH="data/${RUN_CODE}/input.csv"

echo "DATA_FORWARD = $DATA_FORWARD"
echo "DATA_REVERSE = $DATA_REVERSE"
echo "RUN_CODE = $RUN_CODE"
echo "UDI = $UDI"
echo "SUBFILE = $SUBFILE"
echo "ASS_INPUT_FILEPATH = $ASS_INPUT_FILEPATH"
echo "INPUT_FILEPATH = $INPUT_FILEPATH"

cat > $SUBFILE <<EOF
#!/bin/bash

#SBATCH --exclude=ceta3,ceta4,ceta5,ceta6,ceta-gen,ceta.ualg.pt
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=54
#SBATCH --job-name=${UDI}ma
#SBATCH --output=${UDI}ma.log
#SBATCH --error=${UDI}ma.err

export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-17.0.15.0.6-2.el8.x86_64
export PATH=\$JAVA_HOME/bin:\$PATH
export NXF_APPTAINER_CACHEDIR=/share/apps/share/nextflow/apptainer_cache

date
nextflow -log results/assembled/${RUN_CODE}/nexflow.log \\
    run nf-core/mag -r 3.4.0 \\
    -c custom.config \\
    -profile apptainer \\
    --assembly_input data/${RUN_CODE}/assembled_input.csv \\
    --input data/${RUN_CODE}/input.csv \\
    --outdir results/assembled/${RUN_CODE} \\
    --gtdb_db libs/gtdbtk_r202_data.tar.gz \\
    --skip_concoct \\
    --skip_metabat2

#    --skip_gtdbtk
#    -resume
date
EOF

cat > $ASS_INPUT_FILEPATH <<EOF
id,group,assembler,fasta
${UDI},0,MEGAHIT,data/${RUN_CODE}/final.contigs.fa.gz
EOF

cat > $INPUT_FILEPATH <<EOF
sample,group,short_reads_1,short_reads_2,long_reads
${UDI},0,data/${RUN_CODE}/${DATA_FORWARD},data/${RUN_CODE}/${DATA_REVERSE},
EOF

echo
echo "Input file:"
cat $INPUT_FILEPATH
echo
echo "Assembly input file:"
cat $ASS_INPUT_FILEPATH
echo 
echo "Submission script:"
cat $SUBFILE
echo
echo
echo "This nf-core/mag wf uses pre-computed MEGAHIT contigs and does binning using MAXBIN2 only"
```

##### Usage

```
$ ./write-slurm-assembled-script.sh DBH_AABAOSDA_1_1_HCFCYDSX5.UDI129_clean.fastq.gz
```
